In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

products = []
num_pages = 7

for i in range(1, num_pages + 1):

    if i == 1:
        url = "https://www.laptop.lk/?s=&product_cat=laptops&post_type=product"
    else:
        url = f"https://www.laptop.lk/page/{i}/?s&product_cat=laptops&post_type=product"

    print(f"Scraping page: {i}")

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    items = soup.find_all("li", class_="product")

    for item in items:
        try:

            title = item.find("h2", class_="woocommerce-loop-product__title")
            title_text = title.text.strip() if title else "N/A"

            link = item.find("a", class_="woocommerce-LoopProduct-link")
            product_url = link["href"] if link else "N/A"

            image = item.find("img")
            image_url = image["src"] if image else "N/A"

            price = item.find("span", class_="woocommerce-Price-amount")
            price_text = price.text.strip() if price else "N/A"

            description = "N/A"

            if product_url != "N/A":
                product_page = requests.get(product_url, headers=headers)
                product_soup = BeautifulSoup(product_page.text, "html.parser")

                desc_div = product_soup.find(
                    "div",
                    class_="woocommerce-product-details__short-description"
                )

                if desc_div:
                    description = desc_div.get_text(separator=" | ").strip()

                time.sleep(1)  

            products.append({
                "Product Name": title_text,
                "Current Price": price_text,
                "Description": description,
                "Product URL": product_url,
                "Image URL": image_url
            })

            print("Saved:", title_text)

        except Exception as e:
            print("Error scraping product:", e)

    time.sleep(2)

df = pd.DataFrame(products)

for col in ["Current Price", "Old Price"]:
    df[col] = (
        df[col]
        .str.replace("රු", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

df["Current Price"] = pd.to_numeric(df["Current Price"], errors="coerce")

df.to_csv("laptop_data.csv", index=False)
